[![Ouvrir dans Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/philippe-amuzuga/Meca-PY_Ed2026/blob/main/python/P09/P09-1_Mesures_Datees.ipynb)

# Mesures datées

Chapitre 9 du livre *Python pour l'ingénierie mécanique, industrielle et productique*.

## Cadrage

**Contexte industriel.** Le palier d'une machine est suivi depuis quatre ans, un relevé toutes les quatre heures : dix mille lignes où le temps n'est qu'un nombre de jours depuis le début. Votre atelier veut savoir combien de pannes sont survenues en 2018, comment la température a évolué de mois en mois, et si la vibration, très dispersée, dérive vraiment. Vous voudriez dater chaque relevé, regrouper par période, lisser sans effacer la tendance, et recoller les mesures d'un second capteur dont l'horloge est décalée ?

**Objectif terminal.** Ce chapitre montre comment, avec l'index de dates de pandas.

**Objectif instrumental.**

- Donner une date à chaque relevé et sélectionner une période par son nom.
- Regrouper les mesures par jour, par mois ou par année avec `resample`.
- Lisser une série par une fenêtre glissante, la décaler, la différencier.
- Assembler deux séries dont les horloges ne coïncident pas.

**Critère de résultat.** Chaque exercice atteint son résultat attendu, que des contrôles du notebook vérifient.

**Vérification.** Beaucoup d'exercices donnent une seconde voie : un calcul fait par deux chemins indépendants est un calcul qu'on peut croire.

**Prérequis.** Les chapitres 7 et 8, après une filière scientifique.

## Source de données

| Fichier | Lignes | Colonnes | Unités |
|:-|-:|-:|:-|
| `MecaIA-AI4I_maintenance.xlsx` | 10000 | 9 | temps en jours, températures en °C, vitesse en tr/min, couple en N·m, usure en h, puissance en W, vibration en mm/s |
| `MecaIA-AI4I_maintenance_TypePanne.xlsx` | 10000 | 7 | temps en jours, indicateurs 0 ou 1 |

Les classeurs sont lus dans le dossier `donnees` du dépôt `Meca-PY_Ed2026` ; ouvert seul, par exemple dans Colab, le notebook les télécharge une fois depuis le dépôt public.

## Exercices

| N° | Exercice | Durée indicative |
|:-|:-|-:|
| 9.1 | Les pannes de l'année 2018 | 15 min |
| 9.2 | Manquants et moyenne journalière | 15 min |
| 9.3 | Vibration moyenne glissante sur vingt-quatre heures (défi) | 25 min |

Dans le sujet, les cellules marquées `# Votre code :` sont à écrire ; le corrigé les donne résolues. Chaque exercice se termine par des contrôles qui comparent votre résultat à celui du livre.

### Lecture des jeux

Les jeux se cherchent d'abord dans le dépôt cloné, à deux dossiers au-dessus de ce notebook.

In [ ]:
from pathlib import Path
import urllib.request

# Les jeux du chapitre : nom du classeur et adresse de la copie publique.
JEUX = {
    "MecaIA-AI4I_maintenance.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-AI4I_maintenance.xlsx",
    "MecaIA-AI4I_maintenance_TypePanne.xlsx": "https://raw.githubusercontent.com/philippe-amuzuga/Meca-PY_Ed2026/main/donnees/MecaIA-AI4I_maintenance_TypePanne.xlsx",
}

# Le dossier `donnees` du dépôt, cherché depuis le dossier du notebook, puis depuis un clone.
candidats = [Path("..", "..", "donnees"), Path("Meca-PY_Ed2026", "donnees")]
dossier = next((d for d in candidats if all((d / f).is_file() for f in JEUX)), None)
if dossier is None:
    # Notebook ouvert seul (Colab) : copie unique des jeux depuis le dépôt public.
    dossier = Path("donnees")
    dossier.mkdir(exist_ok=True)
    for fichier, adresse in JEUX.items():
        if not (dossier / fichier).is_file():
            try:
                urllib.request.urlretrieve(adresse, dossier / fichier)
            except OSError as erreur:
                raise FileNotFoundError(
                    "Jeu introuvable : clonez le dépôt, ou vérifiez la connexion pour le télécharger.") from erreur
DONNEES = dossier
print("jeux lus dans le dossier des données :", ", ".join(JEUX))

### En-tête de reproductibilité

Versions des bibliothèques employées : les valeurs du livre sont celles de ces versions, avec des graines fixées.

In [ ]:
# En-tête de reproductibilité : versions employées par ce notebook.
import platform
from importlib.metadata import version

print("Python", platform.python_version())
print("pandas", version("pandas"))

### Préparation du dépôt (Colab)

Ouvert seul dans Colab, ce notebook ne voit pas les dossiers du dépôt. La cellule suivante ne télécharge rien : elle cherche `donnees/` et `outils/` (à deux dossiers au-dessus du notebook, à côté de lui ou dans un clone nommé `Meca-PY_Ed2026`), ajoute `outils/` au chemin de recherche de Python, et, sur Colab, liste ce qui manque. Pour le fournir, ouvrez le panneau des fichiers de Colab, créez les dossiers `donnees` et `outils` à la racine de l'espace de travail, et téléversez-y les fichiers listés, pris dans le dépôt. Ce notebook lit les jeux de sa première cellule.

In [ ]:
# Préparation du dépôt (utile sur Colab) : cette cellule ne télécharge rien.
import os
import sys
from pathlib import Path

JEUX_LUS = list(JEUX)                                  # les classeurs de la première cellule
MODULES_DU_LIVRE = []                          # fichiers de outils/


def chercher(dossier):
    """Le dossier du dépôt cherché depuis le notebook, puis depuis un clone ; None s'il est introuvable."""
    for base in (Path("..", ".."), Path("."), Path("Meca-PY_Ed2026")):
        if (base / dossier).is_dir():
            return base / dossier
    return None


donnees, outils = chercher("donnees"), chercher("outils")
if outils is not None and str(outils) not in sys.path:
    sys.path.insert(0, str(outils))
manque = [f"donnees/{nom}" for nom in JEUX_LUS if donnees is None or not (donnees / nom).is_file()]
manque += [f"outils/{nom}" for nom in MODULES_DU_LIVRE if outils is None or not (outils / nom).is_file()]
if manque and "COLAB_RELEASE_TAG" in os.environ:          # variable posée par Colab
    print("À téléverser dans l'espace de travail de Colab, panneau des fichiers :")
    print("\n".join("  " + nom for nom in manque))


### Fonctions du module du livre (`meca.py`)

Le code ci-dessous reprend, tel qu'il est écrit dans le livre, les fonctions du module employées dans ce notebook ; ce code est fourni tel quel.

In [ ]:
import pandas as pd


def indexer_horodatage(df_in, depart, colonne="time", unite="D", arrondi="h", nom="Horodatage"):
    """Remplace une colonne de temps relatif par un index de dates (démarche de l'auteur).

    La colonne `colonne` compte le temps écoulé depuis `depart`, dans l'unité de la fiche du jeu ;
    l'index obtenu est arrondi à `arrondi` puis nommé `nom`.

    >>> d = pd.DataFrame({"time": [0.0, 0.1667, 0.3334], "T": [20.0, 21.0, 22.0]})
    >>> out = indexer_horodatage(d, pd.Timestamp("2025-01-01 08:00"))
    >>> [str(t) for t in out.index]
    ['2025-01-01 08:00:00', '2025-01-01 12:00:00', '2025-01-01 16:00:00']
    >>> out.index.name
    'Horodatage'
    """
    df = df_in.copy()
    df = df.set_index(colonne)
    df.index = depart + pd.to_timedelta(df.index, unit=unite)
    df.index = df.index.round(arrondi)
    df.index.name = nom
    return df

### Préparation

In [ ]:
import pandas as pd
CHEMIN_PANNES = DONNEES / "MecaIA-AI4I_maintenance.xlsx"
CHEMIN_MODES = DONNEES / "MecaIA-AI4I_maintenance_TypePanne.xlsx"
# unité du temps relatif, lue dans la fiche : « jours » devient l'unité "D" de pandas
UNITE_TEMPS = "jours"
UNITE_PANDAS = {"jours": "D", "heures": "h", "minutes": "min", "secondes": "s"}["jours"]
pd.set_option("display.width", 100)

### Section 9.1 : un index horodaté ordonne les mesures.

In [ ]:
print("unité du temps lue dans la fiche du jeu :", "jours", "->", UNITE_PANDAS)
# notion: horodatage
depart = pd.Timestamp("2017-07-07 08:00")
grille = pd.date_range(depart, periods=5, freq="4h")                 # alias en minuscule : "h"
pas = pd.Timedelta(hours=4)
durees = pd.to_timedelta([0.0, 0.1667, 0.3334], unit="D")
assert grille[1] - grille[0] == pas and len(grille) == 5
assert (
    abs(durees[1] - pas) < pd.Timedelta(seconds=5)
), "0,1667 jour vaut quatre heures à trois secondes près"
print("0,1667 jour =", durees[1].round("s"), "; écart à quatre heures :", abs(durees[1] - pas).round("s"))

In [ ]:
# notion: datetime-index
dates = pd.to_datetime(["2017-07-07 08:00", "2018-03-01 00:00", "2018-12-31 20:00", "2019-01-01 00:00"])
serie = pd.Series([1, 2, 3, 4], index=dates)
assert isinstance(serie.index, pd.DatetimeIndex)
assert serie["2018"].tolist() == [2, 3], "une année"
assert serie["2018-03"].tolist() == [2], "un mois"
assert serie["2017-07-07":"2018-03-01"].tolist() == [1, 2], "un intervalle"

## Exercice 9.1 : Les pannes de l'année 2018 (15 min)

Donnez à chaque relevé du suivi du palier sa date : départ le 7 juillet 2017 à 8 h, durée lue dans la colonne `time` avec l'unité de la fiche, arrondie à l'heure. Combien de pannes (colonne `Panne` égale à $1$) sont survenues en 2018 ?

**Résultat attendu.** Premier relevé le 2017-07-07, dernier le 2022-01-29 ; 54 pannes en 2018.

**Seconde voie.** Un masque sur l'année de l'index, `df.index.year == 2018`, donne le même compte.

**Indication.** Copiez le classeur, passez `time` en index, ajoutez à l'instant de départ la durée `pd.to_timedelta` de l'index, avec `unit="D"`, puis arrondissez avec `round("h")`. Le compte est la somme de la colonne `Panne` sur `df.loc["2018"]`. Le masque `df.index.year == 2018` redonne le même résultat.

Les contrôles de la cellule lisent les variables `data`, `df`, `pannes_2018` : gardez ces noms.

In [ ]:
# Votre code :
assert (
    df.index.is_unique and df.index.is_monotonic_increasing
), "un index de dates sans doublon, dans l'ordre"
assert (
    str(df.index.min()) == "2017-07-07 08:00:00" and str(df.index.max()) == "2022-01-29 04:00:00"
)
assert (
    indexer_horodatage(data, depart, unite=UNITE_PANDAS).equals(df)
), "la fonction du module refait ces lignes"
assert (
    pannes_2018 == int(data.loc[(df.index.year == 2018), "Panne"].sum())
), "même compte par un masque"
assert pannes_2018 == 54, "pannes de 2018 attendues : 54"
print("Exercice 9.1 : contrôles passés")

### Section 9.2 : rééchantillonner regroupe par période.

In [ ]:
# notion: resample
temperature = df["T_palier_C"]
manquants_avant = int(temperature.isna().sum())
par_jour = temperature.resample("D").mean()                            # une moyenne par jour
par_mois = temperature.resample("ME").agg(["min", "mean", "max"])      # fin de mois, trois statistiques
print(par_mois.head(3).round(1))
assert (
    len(par_jour) == 1668 and len(par_mois) == 55
), "1 668 jours, 55 mois de juillet 2017 à janvier 2022"
assert (par_mois["min"] <= par_mois["mean"]).all() and (par_mois["mean"] <= par_mois["max"]).all()

## Exercice 9.2 : Manquants et moyenne journalière (15 min)

La température du palier compte des relevés manquants. Combien ? Résumez-la par une moyenne journalière avec `resample("D")` : combien de jours restent sans valeur ?

**Résultat attendu.** 196 relevés manquants ; 0 jour sans valeur sur 1 668.

**Seconde voie.** Compter les jours qui portent au moins un relevé valide, avec `dropna`, puis `index.normalize()`, puis `nunique()`, redonne 1 668 jours : aucun jour n'est vide.

**Indication.** Le compte des manquants est `temperature.isna().sum()`. La moyenne journalière est `temperature.resample("D").mean()` ; `isna().sum()` sur ce résultat compte les jours vides. Chaque jour garde au moins un relevé valide.

Les contrôles de la cellule lisent la variable `jours_sans_valeur` : gardez ce nom.

In [ ]:
# Votre code :
assert manquants_avant == 196, "valeurs manquantes attendues : 196"
assert jours_sans_valeur == 0, "aucun jour ne reste sans valeur"
print("Exercice 9.2 : contrôles passés")

### Section 9.3 : une fenêtre glissante lisse et assemble.

In [ ]:
vibration = df["Vibration_mm_s"]
# notion: fenetres
pas_irreguliers = int((df.index.to_series().diff() != pd.Timedelta(hours=4)).sum()) - 1     # le premier est vide
print("pas différents de quatre heures :", pas_irreguliers)
assert (
    pas_irreguliers == 8
), "huit pas de cinq heures : le 0,1667 jour de la fiche n'est pas exactement quatre heures"
moyenne_mobile = vibration.rolling(window=6, min_periods=1).mean()     # six relevés de quatre heures : 24 h
lissage_exp = vibration.ewm(alpha=0.5).mean()
precedent = vibration.shift(1)
variation = vibration.diff()
assert (
    moyenne_mobile.iloc[:3].tolist() == [vibration.iloc[:1].mean(), vibration.iloc[:2].mean(), vibration.iloc[:3].mean()]
)

In [ ]:
assert (
    abs(lissage_exp.iloc[0] - vibration.iloc[0]) < 1e-12
), "l'exponentielle démarre sur la première valeur"
assert (
    precedent.iloc[1] == vibration.iloc[0] and abs(variation.iloc[1] - (vibration.iloc[1] - vibration.iloc[0])) < 1e-12
)

In [ ]:
# notion: merge-asof
modes = pd.read_excel(CHEMIN_MODES)
decalage = pd.Timedelta(minutes=70)
second = modes.set_index("time")
second.index = depart + decalage + pd.to_timedelta(second.index, unit=UNITE_PANDAS)
second.index.name = "Horodatage"
gauche = df[["Vibration_mm_s", "Panne"]].reset_index()
droite = second[["Panne_outil", "Panne_surcharge"]].reset_index()
assemble = pd.merge_asof(gauche, droite, on="Horodatage", direction="nearest", tolerance=pd.Timedelta(hours=2))
strict = pd.merge_asof(gauche, droite, on="Horodatage", direction="nearest", tolerance=pd.Timedelta(minutes=30))
trouves = int(assemble["Panne_outil"].notna().sum())
trouves_strict = int(strict["Panne_outil"].notna().sum())
print(f"seconde source décalée de {decalage} : {trouves} relevés appariés à moins de 2 h, {trouves_strict} à moins de 30 min")
assert assemble.shape[0] == df.shape[0], "merge_asof garde toutes les lignes de la série de gauche"

In [ ]:
assert (
    trouves == 10000 and trouves_strict == 0
), "tolérance de 2 h : tous appariés ; de 30 min : aucun"

## Exercice 9.3 : Vibration moyenne glissante sur vingt-quatre heures (défi, 25 min)

Calculez la moyenne glissante de la vibration sur six relevés complets, c'est-à-dire sans `min_periods`. Quelle est sa valeur maximale, et à quelle date ? Comparez-la au maximum des relevés bruts.

**Résultat attendu.** Maximum de 19,123 mm/s le 2019-12-24 ; le maximum brut, 20,0 mm/s, reste plus élevé.

**Seconde voie.** Une convolution par un vecteur de six coefficients égaux à $1/6$, suivie de `np.nanmax`, redonne la même valeur.

**Indication.** La moyenne glissante est `rolling` avec `window=6`, suivie de `mean`. Son maximum est `max`, sa date `idxmax`. Une moyenne ne dépasse jamais le plus grand des relevés qu'elle moyenne : elle reste sous le maximum brut. La convolution `np.convolve` en mode `"valid"` donne les mêmes moyennes, sans index, et `np.nanmax` ignore les fenêtres vides.

Les contrôles de la cellule lisent les variables `date_maximum`, `glissante`, `maximum` : gardez ces noms.

In [ ]:
# Votre code :
assert abs(maximum - 19.123) < 0.0005, "maximum attendu : 19,123 mm/s"
assert str(date_maximum) == "2019-12-24 12:00:00", "date du maximum attendue : 2019-12-24 12:00"
assert maximum < vibration.max(), "une moyenne ne dépasse jamais le maximum des mesures"
assert glissante.iloc[:5].isna().all(), "les cinq premières fenêtres sont incomplètes"
print("Exercice 9.3 : contrôles passés")

## Essais des fiches de fonction du chapitre

Chaque fiche du livre donne l'appel minimal d'une fonction. Les cellules suivantes rejouent ces appels, avec les mêmes lignes de code et la même sortie que la fiche : changez une valeur et relisez le résultat.

### Fiche F09-1 : Rééchantillonnage (`pandas.DataFrame.resample`)

In [ ]:
import numpy as np
import pandas as pd

temps = pd.date_range("2024-01-01", periods=3600, freq="s")
# capteur défaillant pendant cinq secondes
vent = pd.DataFrame({"v": 5 + np.random.default_rng(0).normal(0, 1, 3600)},
                    index=temps).drop(temps[10:15])
moy = vent.resample("10min").mean()
print(f"{moy.iloc[0, 0]:.3f}".replace(".", ","))
verif = vent.loc[:"2024-01-01 00:09:59", "v"].mean()
print(np.isclose(moy.iloc[0, 0], verif))      # contrôle du premier intervalle